In [ ]:
## import dependencies
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import pandas as pd
import torch
import plotly
import requests
#import STalign
#from STalign import STalign
import pickle
import json
import cv2
import scanpy as sc
import anndata as ad
import muon

# make plots bigger
plt.rcParams["figure.figsize"] = (12,10)
plt.rcParams["savefig.dpi"] = 300

### Plots at the original resolution

In [ ]:
he_id = 'V11L12-109_B1'
sample_id = he_id+'_plusplus_sub'
sample_id2 = he_id+'_plusplus_sub2_norm'

In [ ]:
adata_gene = sc.read_h5ad(f'{he_id}_RNA_joint_cluster_with_hires_img.h5ad')
adata_msi = sc.read_h5ad(f'{sample_id}_MALDI_adata_with_hires_img.h5ad')
adata_msi2 = sc.read_h5ad(f'{sample_id2}_MALDI_adata.h5ad')

In [ ]:
adata_msi2.var_names

In [ ]:
adata_gene

In [ ]:
import squidpy as sq
adata = sq.read.visium(path = '../MALDI/sma/V11L12-109/V11L12-109_B1/output_data/V11L12-109_B1_RNA/outs/')

In [ ]:
adata.var_names_make_unique()
adata_sub = adata[:, adata_gene.var_names].copy()

In [ ]:
adata_sub

In [ ]:
sc.pp.normalize_total(adata_sub, target_sum=1e4)
sc.pp.log1p(adata_sub)
#sc.pp.highly_variable_genes(adata, n_top_genes=500)

# scale data, clip values exceeding standard deviation 10.
sc.pp.scale(adata_sub, max_value=10)

sc.tl.pca(adata_sub, svd_solver='arpack')
# plot more PCS
sc.pl.pca(adata_sub, components = ['1,2','3,4','5,6','7,8'], ncols=2)

In [ ]:
# Then we calculate a new umap
sc.pp.neighbors(adata_sub, n_neighbors=50, n_pcs=25, use_rep='X_pca', metric='cosine', random_state=0)

sc.tl.umap(adata_sub, min_dist=0.0, random_state=0)

In [ ]:
sc.tl.leiden(adata_sub, resolution = 0.5, key_added = "leiden_0.5", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(adata_sub, resolution = 0.6, key_added = "leiden_0.6", random_state=0, flavor='igraph', n_iterations=2)
sc.pl.umap(adata_sub, color='leiden_0.6')

In [ ]:
sc.pl.spatial(adata_sub, color='leiden_0.6', img_key='hires', size=1.0)

In [ ]:
from PIL import Image, ImageOps
xenium_image = Image.fromarray(adata_sub.uns['spatial']['V11L12-109_B1']['images']['hires'].copy())
invert_xenium_image = ImageOps.invert(xenium_image)
gray_xenium_image = invert_xenium_image.convert('LA')

gray_xenium_image_array = np.array(gray_xenium_image)
gray_xenium_image_final = np.stack((gray_xenium_image_array[:, :, 0]/255.,)*3, axis=-1)
plt.imshow(gray_xenium_image_final, origin='lower')

In [ ]:
threshold = 100/255.  # Define your threshold
# Create a copy so you don't lose the original data
processed_array = gray_xenium_image_final.copy()

# Set all values less than the threshold to 0
processed_array[processed_array < threshold] = 0

plt.imshow(processed_array, origin='lower')

In [ ]:
from scipy.ndimage import rotate
img_rotated = rotate(processed_array, angle=30, reshape=False, order=1)

# Plot to verify
plt.imshow(img_rotated)

In [ ]:
def rotate_around_center(x, y, deg, center_x, center_y):
    # 1. Calculate the center (centroid) of the points
    #center_x = np.mean(x)
    #center_y = np.mean(y)
    
    # 2. Shift points so the center is at (0,0)
    x_shifted = x - center_x
    y_shifted = y - center_y
    
    # 3. Define rotation angle in radians
    # Counter-clockwise rotation: use +theta
    # Clockwise rotation: use -theta
    theta = np.radians(deg)
    
    # 4. Rotation matrix
    c, s = np.cos(theta), np.sin(theta)
    # Standard 2D rotation matrix:
    # x' = x*cos(theta) - y*sin(theta)
    # y' = x*sin(theta) + y*cos(theta)
    
    x_new = x_shifted * c - y_shifted * s
    y_new = x_shifted * s + y_shifted * c
    
    # 5. Shift points back to the original center
    x_final = x_new + center_x
    y_final = y_new + center_y
    
    return x_final, y_final

In [ ]:
adata_sub.obsm['spatial_origin'] = adata_sub.obsm['spatial'].copy()

In [ ]:
adata_sub.uns['spatial']['V11L12-109_B1']['images']['hires_30rot'] = img_rotated.copy()
adata_sub.uns['spatial']['V11L12-109_B1']['scalefactors']['tissue_hires_30rot_scalef'] = (adata_msi.uns['spatial'][sample_id]['scalefactors']['tissue_hires_scalef']/1.265).copy()

In [ ]:
rot_x_30, rot_y_30 = rotate_around_center(adata_sub.obsm['spatial_origin'][:, 0], adata_sub.obsm['spatial_origin'][:, 1], -30, np.mean(adata_sub.obsm['spatial_origin'][:, 0]), np.mean(adata_sub.obsm['spatial_origin'][:, 1]))
adata_sub.obsm['spatial_30_rot'] = np.array([rot_x_30+1000, rot_y_30+1000]).T
adata_sub.obsm['spatial'] = adata_sub.obsm['spatial_30_rot'].copy()

In [ ]:
sc.pl.spatial(adata_sub, color='leiden_0.6', img_key='hires_30rot', size=1.0)

### add MALDI spots

In [ ]:
he_dir = f'/gpfs/ycga/work/fan/yl2499/CellViT/maldi_mousebrain/preprocessing/MSi_{he_id}/MSi_{he_id}/cell_detection'
msi_fmp = pd.read_csv('../MALDI/sma/V11L12-109/V11L12-109_B1/output_data/V11L12-109_B1_MSI/V11L12-109_B1.Visium.FMP.220826_smamsi.csv')
in_tissue_spots = np.sum(msi_fmp.iloc[:, 2:]>0, axis=1)>0
msi_fmp_sub = msi_fmp.loc[in_tissue_spots, :].copy()

In [ ]:
plt.imshow(adata_sub.uns['spatial']['V11L12-109_B1']['images']['hires'], origin='lower')

In [ ]:
plt.scatter(adata_gene.obsm['spatial'][:, 0], adata_gene.obsm['spatial'][:, 1])

In [ ]:
np.max(msi_fmp.y)

In [ ]:
plt.scatter(-msi_fmp.y, msi_fmp.x, c=np.sum(msi_fmp.iloc[:, 2:]>0, axis=1))

In [ ]:
def get_bin_centers(start, end, num_bins):
    # Create the edges (num_bins + 1 edges create num_bins spaces)
    edges = np.linspace(start, end, num_bins + 1)
    
    # Calculate centers: (edge_i + edge_i+1) / 2
    centers = (edges[:-1] + edges[1:]) / 2
    return centers

In [ ]:
maldi_spots_centers_x = get_bin_centers(np.min(adata_gene.obsm['spatial'][:, 0]), np.max(adata_gene.obsm['spatial'][:, 0]), 78)
maldi_spots_centers_y = get_bin_centers(np.min(adata_gene.obsm['spatial'][:, 1]), np.max(adata_gene.obsm['spatial'][:, 1]), 63)

In [ ]:
maldi_spots_centers_x_full = maldi_spots_centers_x[77-msi_fmp_sub['y']]
maldi_spots_centers_y_full = maldi_spots_centers_y[msi_fmp_sub['x']]

In [ ]:
plt.scatter(maldi_spots_centers_x_full, maldi_spots_centers_y_full)

In [ ]:
### calculate number of cells in each MALDI spot
sample_id = 'V11L12-109_B1_plusplus_sub'
with open(f'./{sample_id}_full_data_level1.pickle', 'rb') as file:
    cellvit_marker_array, other_marker_array_sub, gene_exp = pickle.load(file)

In [ ]:
other_marker_array_sub

In [ ]:
import anndata as ad
from scipy import sparse
msi_fmp_sub_values = msi_fmp_sub.iloc[:, 2:].copy()
msi_fmp_sub_clipped = msi_fmp_sub_values.clip(upper=msi_fmp_sub_values.quantile(0.995), axis=1)
msi_fmp_sub_clipped_norm = (msi_fmp_sub_clipped / msi_fmp_sub_clipped.max()) * 100
adata_maldi = ad.AnnData(sparse.csc_matrix(msi_fmp_sub_clipped_norm.values+1))
adata_maldi.var_names = list(msi_fmp_sub_clipped.columns)
adata_maldi.obs_names = np.array(msi_fmp_sub_clipped.index).astype(str)
adata_maldi.obs['array_col'] = np.array(77-msi_fmp_sub['y'])
adata_maldi.obs['array_row'] = np.array(msi_fmp_sub['x'])
adata_maldi.obs['x'] = np.array(maldi_spots_centers_x_full)
adata_maldi.obs['y'] = np.array(maldi_spots_centers_y_full)
adata_maldi.obsm['spatial'] = np.array([adata_maldi.obs['x'], adata_maldi.obs['y']]).T
adata_maldi.obsm['spatial_origin'] = adata_maldi.obsm['spatial'].copy()

In [ ]:
adata_maldi.uns['spatial'] = adata_sub.uns['spatial'].copy()
adata_maldi.uns['spatial'][he_id]['scalefactors']['spot_diameter_fullres'] = 564.4

In [ ]:
sc.pl.spatial(adata_maldi, color='674.2805', img_key='hires', size=1.0)

In [ ]:
adata_maldi.uns['spatial']['V11L12-109_B1']['images']['hires_30rot'] = img_rotated.copy()
adata_maldi.uns['spatial']['V11L12-109_B1']['scalefactors']['tissue_hires_30rot_scalef'] = (adata_msi.uns['spatial'][sample_id]['scalefactors']['tissue_hires_scalef']).copy()
rot_x_30, rot_y_30 = rotate_around_center(adata_maldi.obsm['spatial_origin'][:, 0], adata_maldi.obsm['spatial_origin'][:, 1], -30, np.mean(adata_maldi.obsm['spatial_origin'][:, 0]), np.mean(adata_maldi.obsm['spatial_origin'][:, 1]))
adata_maldi.obsm['spatial_30_rot'] = (np.array([rot_x_30, rot_y_30-300]).T/1.21).copy()
adata_maldi.obsm['spatial'] = adata_maldi.obsm['spatial_30_rot'].copy()

In [ ]:
adata_msi.var_names

In [ ]:
selected_maldi_channel=['353.1652','371.17565','393.12703','421.19136','423.18201', '435.20692','444.20715','674.2805','698.49196']

In [ ]:
adata_maldi_sub = adata_maldi[other_marker_array_sub.Barcode, :].copy()
adata_maldi_sub.obs['n_cells'] = np.array(cellvit_marker_array[:, -9]).copy()
norm_signal = (adata_maldi_sub.X.toarray()/(np.array(adata_maldi_sub.obs['n_cells'])[:, None])).copy()
adata_maldi_sub.layers['norm_signal'] = norm_signal

In [ ]:
selected_maldi_channel

In [ ]:
for peak in selected_maldi_channel:
    sc.pl.spatial(adata_maldi_sub, color=peak, img_key='hires_30rot', layer='norm_signal', size=1.0, alpha=1.0, frameon=False, legend_loc=None, title='', show=False, cmap='turbo', colorbar_loc=None, 
                 vmax=peak_max[np.where(adata_maldi_sub.var_names == peak)[0][0]]*0.5)

In [ ]:
leiden_cluster = ['t'+c for c in adata_sub.obs['leiden_0.6']]
adata_sub.obs['leiden_cluster'] = leiden_cluster

In [ ]:
ax = sc.pl.spatial(adata_sub, color='leiden_cluster', img_key='hires_30rot', size=1.0, alpha=1.0, frameon=False, show=False)
legend = plt.legend(
        bbox_to_anchor=(0.95, 0.9), 
        loc='upper left', 
        frameon=False,
        fontsize=19,          # Explicitly set legend font size
        title_fontsize=15,     # Set legend title size if applicable
        markerscale=2.0
    )

In [ ]:
adata_sub.obs['visium_spot'] = ['Visium']*adata_sub.n_obs
adata_maldi.obs['maldi_spot'] = ['Maldi']*adata_maldi.n_obs

In [ ]:
adata_maldi.uns['maldi_spot_colors'] = ['#ff7f0e']#'#ff7f0e'

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# Create a single figure and axis
fig, ax = plt.subplots(figsize=(10, 10))

# 1. Plot the RNA Leiden Clusters as the base layer
sc.pl.spatial(adata_sub, 
              color='visium_spot', 
              img_key='hires_30rot', 
              size=1.0, 
              ax=ax, 
              alpha_img=0,
              alpha=1.0, 
              show=False, frameon=False)

# 2. Overlay the MALDI peak 674.2805 from cell [90]
# We use alpha=0.6 so we can see the clusters underneath
sc.pl.spatial(adata_maldi, 
              color='maldi_spot', 
              img_key='hires_30rot', # Set to None so it doesn't plot the background image twice
              size=1.0, 
              cmap='turbo',
              alpha_img=0,
              alpha=0.4, 
              ax=ax, 
              show=False, frameon=False)
ax.invert_yaxis()
ax.set_title('Overlay: Visium & MALDI spots', fontsize=20)
plt.show()

In [ ]:
plt.scatter(adata_sub.obsm['spatial'][:, 0], adata_sub.obsm['spatial'][:, 1])
plt.scatter(adata_maldi.obsm['spatial'][:, 0], adata_maldi.obsm['spatial'][:, 1])

In [ ]:
adata_sub_sub = adata_sub[(adata_sub.obsm['spatial'][:, 0]>30000)*(adata_sub.obsm['spatial'][:, 0]<35000)*(adata_sub.obsm['spatial'][:, 1]>20000)*(adata_sub.obsm['spatial'][:, 1]<25000), :].copy()

In [ ]:
adata_maldi_sub

In [ ]:
# Create a single figure and axis
fig, ax = plt.subplots(figsize=(10, 10))

# 1. Plot the RNA Leiden Clusters as the base layer
sc.pl.spatial(adata_sub_sub, 
              color='visium_spot', 
              img_key='hires_30rot', 
              size=1.0, 
              ax=ax, 
              alpha_img=0,
              alpha=1.0, 
              show=False, frameon=False)

# 2. Overlay the MALDI peak 674.2805 from cell [90]
# We use alpha=0.6 so we can see the clusters underneath
sc.pl.spatial(adata_maldi, 
              color='maldi_spot', 
              img_key='hires_30rot', # Set to None so it doesn't plot the background image twice
              size=0.85, 
              cmap='turbo',
              alpha_img=0,
              alpha=0.5, 
              marker='s',
              ax=ax, 
              show=False, frameon=False)
ax.invert_yaxis()
legend = plt.legend(
        bbox_to_anchor=(0.93, 0.9), 
        loc='upper left', 
        frameon=False,
        fontsize=25,          # Explicitly set legend font size
        title_fontsize=15,     # Set legend title size if applicable
        markerscale=2.0
    )
ax.set_title('Overlay: Visium & MALDI spots', fontsize=20)
plt.show()

In [ ]:
adata_maldi_sub = adata_maldi[:, adata_msi2.var_names].copy()

In [ ]:
adata_maldi_sub.layers['counts'] = adata_maldi_sub.X.copy()

In [ ]:
sc.pp.normalize_total(adata_maldi_sub, target_sum=1e4)
sc.pp.log1p(adata_maldi_sub)
#sc.pp.highly_variable_genes(adata, n_top_genes=500)

# scale data, clip values exceeding standard deviation 10.
sc.pp.scale(adata_maldi_sub, max_value=10)

sc.tl.pca(adata_maldi_sub, svd_solver='arpack')
# plot more PCS
sc.pl.pca(adata_maldi_sub, components = ['1,2','3,4','5,6','7,8'], ncols=2)

In [ ]:
# Then we calculate a new umap
sc.pp.neighbors(adata_maldi_sub, n_neighbors=50, n_pcs=15, use_rep='X_pca', metric='correlation', random_state=0)

sc.tl.umap(adata_maldi_sub, min_dist=0.0, random_state=0)

In [ ]:
sc.tl.leiden(adata_maldi_sub, resolution = 0.3, key_added = "leiden_0.3_sc", random_state=0, flavor='igraph', n_iterations=2)

In [ ]:
leiden_cluster = ['m'+c for c in adata_maldi_sub.obs['leiden_0.3_sc']]
adata_maldi_sub.obs['leiden_cluster'] = leiden_cluster

In [ ]:
ax = sc.pl.spatial(adata_maldi_sub, color='leiden_cluster', img_key='hires_30rot', size=1.0, alpha=0.5, frameon=False, show=False)
legend = plt.legend(
        bbox_to_anchor=(0.92, 0.9), 
        loc='upper left', 
        frameon=False,
        fontsize=19,          # Explicitly set legend font size
        title_fontsize=15,     # Set legend title size if applicable
        markerscale=2.0
    )

In [ ]:
adata_gene = sc.read_h5ad(f'{he_id}_RNA_joint_cluster_with_hires_img.h5ad')
adata_msi = sc.read_h5ad(f'{sample_id}_MALDI_adata_with_hires_img.h5ad')
adata_msi2 = sc.read_h5ad(f'{sample_id2}_MALDI_adata.h5ad')

In [ ]:
from scipy.ndimage import rotate
img = adata_msi.uns['spatial'][sample_id]['images']['hires'].copy()
# Rotate the image array by 30 degrees
# reshape=False keeps the dimensions the same
# order=1: uses bi-linear interpolation (smoother than the default order=3 for large images)
# mode='constant', cval=0: fills the 'empty' corners with black
img_rotated = rotate(img, angle=30, reshape=False, order=1)

# Plot to verify
plt.imshow(img_rotated)

In [ ]:
adata_msi.uns['spatial'][sample_id]['images']['hires_30rot'] = img_rotated.copy()
adata_msi.uns['spatial'][sample_id]['scalefactors']['tissue_hires_30rot_scalef'] = adata_msi.uns['spatial'][sample_id]['scalefactors']['tissue_hires_scalef']
rot_x_30, rot_y_30 = rotate_around_center(adata_msi.obsm['spatial_origin'][:, 0], adata_msi.obsm['spatial_origin'][:, 1], -30, 47616/2, 48128/2)
adata_msi.obsm['spatial_30_rot'] = np.array([rot_x_30-50, rot_y_30-220]).T

In [ ]:
adata_msi

In [ ]:
#plt.imshow(adata_msi2.uns['spatial'][sample_id2]['images']['hires_30rot'])

In [ ]:
adata_msi2.obsm['spatial_origin'] = adata_msi.obsm['spatial_origin'].copy()
adata_msi2.uns['spatial'][sample_id2] = adata_msi.uns['spatial'][sample_id].copy()

In [ ]:
adata_gene.obsm['spatial'] = adata_msi.obsm['spatial_30_rot'].copy()
adata_msi.obsm['spatial'] = adata_msi.obsm['spatial_30_rot'].copy()
adata_msi2.obsm['spatial'] = adata_msi.obsm['spatial_30_rot'].copy()

In [ ]:
adata_msi2.uns['leiden_0.3_sc_colors'] = ['#00FFFF', #'#1f77b4', 
                                          '#aec7e8', '#279e68', '#d62728', '#FFFF00', #'#aa40fc', 
                                          '#8c564b', '#e377c2', '#0000FF', #'#b5bd61', 
                                          '#17becf', '#ff7f0e', '#ffbb78', '#98df8a']

In [ ]:
leiden_cluster = ['m'+c for c in adata_msi2.obs['leiden_0.3_sc']]
adata_msi2.obs['leiden_cluster'] = leiden_cluster
adata_msi2.uns['leiden_cluster_colors'] = ['#00FFFF', #'#1f77b4', 
                                          '#aec7e8', '#279e68', '#d62728', '#FFFF00', #'#aa40fc', 
                                          '#8c564b', '#e377c2', '#0000FF', #'#b5bd61', 
                                          '#17becf', '#ff7f0e', '#ffbb78', '#98df8a']
sc.pl.spatial(adata_msi2, color='leiden_cluster', img_key='hires_30rot', size=6, alpha_img=1.0, frameon=False, show=False)
legend = plt.legend(
        bbox_to_anchor=(0.93, 1.05), 
        loc='upper left', 
        frameon=False,
        fontsize=20,          # Explicitly set legend font size
        title_fontsize=15,     # Set legend title size if applicable
        markerscale=2.0
    )

In [ ]:
fig = sc.pl.spatial(adata_msi2, color='444.20715', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo', frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.1, vmax=1.0, colorbar_loc=None)
#fig.savefig(f'{sample_id}_maldi_Serotonin_peak.pdf', bbox_inches='tight', pad_inches=0)

In [ ]:
fig = sc.pl.spatial(adata_msi2, color='674.2805', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmax=3.5, colorbar_loc=None)
#fig.savefig(f'{sample_id}_maldi_Dopamine_DD_peak.pdf', bbox_inches='tight', pad_inches=0)

In [ ]:
fig = sc.pl.spatial(adata_msi2, color='393.12703', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.1, vmax = 0.8, colorbar_loc=None)
#fig.savefig(f'{sample_id}_maldi_Taurine_peak.pdf', bbox_inches='tight', pad_inches=0)

In [ ]:
fig = sc.pl.spatial(adata_msi2, color='421.19136', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.02, vmax = 1.5, colorbar_loc=None)
#fig.savefig(f'{sample_id}_maldi_Dopamine_peak.pdf', bbox_inches='tight', pad_inches=0)

In [ ]:
fig = sc.pl.spatial(adata_msi2, color='423.18201', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.2, vmax = 1.0, colorbar_loc=None)
#fig.savefig(f'{sample_id}_maldi_Histidine_peak.pdf', bbox_inches='tight', pad_inches=0)

In [ ]:
fig = sc.pl.spatial(adata_msi2, color='435.20692', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.02,vmax=4.0,colorbar_loc=None)
#fig.savefig(f'{sample_id}_maldi_3-MT_peak.pdf', bbox_inches='tight', pad_inches=0)

In [ ]:
fig = sc.pl.spatial(adata_msi2, color='698.49196', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.02, vmax=1.0, colorbar_loc=None)
#fig.savefig(f'{sample_id}_maldi_Tocopherol_peak.pdf', bbox_inches='tight', pad_inches=0)

In [ ]:
fig = sc.pl.spatial(adata_msi2, color='353.1652', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.02, vmax=1.0, colorbar_loc=None)
#fig.savefig(f'{sample_id}_maldi_GABA-H2O_peak.pdf', bbox_inches='tight', pad_inches=0)

In [ ]:
fig = sc.pl.spatial(adata_msi2, color='371.17565', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.1, vmax=0.7, colorbar_loc=None)
#fig.savefig(f'{sample_id}_maldi_GABA_peak.pdf', bbox_inches='tight', pad_inches=0)

In [ ]:
adata_msi2.var_names

In [ ]:
adata_gene.uns['spatial'][sample_id]['images']['hires_30rot'] = img_rotated.copy()
adata_gene.uns['spatial'][sample_id]['scalefactors']['tissue_hires_30rot_scalef'] = adata_gene.uns['spatial'][sample_id]['scalefactors']['tissue_hires_scalef']
adata_gene.obsm['spatial_30_rot'] = np.array([rot_x_30-50, rot_y_30-220]).T

In [ ]:
leiden_cluster = ['t'+c for c in adata_gene.obs['leiden_0.5']]
adata_gene.obs['leiden_cluster'] = leiden_cluster

adata_gene.uns['leiden_cluster_colors'] = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#bcbd22',
       '#e377c2', '#ffbb78', '#8c564b', '#17becf']
sc.pl.spatial(adata_gene, color='leiden_cluster', img_key='hires_30rot', size=6, alpha_img=1.0, frameon=False, show=False)
legend = plt.legend(
        bbox_to_anchor=(0.93, 0.9), 
        loc='upper left', 
        frameon=False,
        fontsize=20,          # Explicitly set legend font size
        title_fontsize=15,     # Set legend title size if applicable
        markerscale=2.0
    )

In [ ]:
adata_gene.uns['leiden_0.5_colors'] = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#bcbd22',
       '#e377c2', '#ffbb78', '#8c564b', '#17becf']

In [ ]:
sc.pl.spatial(adata_gene, color='leiden_0.5', library_id=sample_id, img_key='hires_30rot', size=5, alpha_img=1.0, frameon=False)

In [ ]:
adata_gene.uns['cell_type_colors'] = ['#1f77b4', '#ff7f0e', '#279e68', '#d62728', '#aa40fc', '#8c564b',
       '#e377c2', '#b5bd61', '#17becf', '#aec7e8', '#ffbb78', '#98df8a',
       '#ff9896', '#c5b0d5', '#fcf4a3']#'#c49c94']

In [ ]:
adata_gene

In [ ]:
adata_gene2 = sc.read_h5ad(f'{he_id}_RNA_norm_maldi_joint_cluster.h5ad')

In [ ]:
adata_gene.obs['leiden_0.5_joint'] = adata_gene2.obs['leiden_0.5_joint'].copy()

In [ ]:
sc.pl.spatial(adata_gene, color='leiden_0.5_joint', library_id=sample_id, img_key='hires_30rot', size=5, alpha_img=1.0, frameon=False)
sc.pl.spatial(adata_gene, color='leiden_0.4_joint', library_id=sample_id, img_key='hires_30rot', size=5, alpha_img=1.0, frameon=False)

In [ ]:
adata_gene.uns['cell_type_colors'] = ['#1f77b4',
 '#ff7f0e',
 '#279e68',
 '#d62728',
 '#aa40fc',
 '#66FF00',#'#8c564b',
 '#e377c2',
 '#b5bd61',
 '#17becf',
 '#aec7e8',
 '#ffbb78',
 '#98df8a',
 '#ff9896',
 '#c5b0d5',
 '#fcf4a3']

In [ ]:
# Define the mapping based on the cluster markers
cluster_to_celltype = {
    '0': 'Glutamatergic Neurons',
    '1': 'Large Projecting Neurons',
    '2': 'Endothelial Cells',
    '3': 'iSPN',
    '4': 'Immune Associated Cells',
    '5': 'Oligodendrocytes',
    '6': 'Deep Layer Cortical Neurons',
    '7': 'Interneurons',
    '8': 'dSPN',
    '9': 'Astrocytes',
    '10': 'Developing Striatal Neurons',
    '11': 'GABAergic Interneurons',
    '12': 'Striatal/Border Cells',
    '13': 'Ependymal Cells',
    '14': 'Microglia'
}

# Create the new 'cell_type' column in adata.obs
adata_gene.obs['cell_type_final'] = adata_gene.obs['leiden_0.5_joint'].map(cluster_to_celltype)

# Verify the assignment
print(adata_gene.obs['cell_type_final'].value_counts())

In [ ]:
adata_gene.uns['cell_type_final_colors'] = [adata_gene.uns['cell_type_colors'][8], 
                                           adata_gene.uns['cell_type_colors'][10], 
                                           adata_gene.uns['cell_type_colors'][9], 
                                           adata_gene.uns['cell_type_colors'][2], 
                                           adata_gene.uns['leiden_0.5_joint_colors'][4], 
                                           adata_gene.uns['cell_type_colors'][14],
                                           adata_gene.uns['cell_type_colors'][11], 
                                           adata_gene.uns['cell_type_colors'][6], 
                                           adata_gene.uns['cell_type_colors'][1], 
                                           adata_gene.uns['cell_type_colors'][3], 
                                           adata_gene.uns['cell_type_colors'][13], 
                                           adata_gene.uns['cell_type_colors'][7], 
                                           adata_gene.uns['cell_type_colors'][0],
                                           adata_gene.uns['cell_type_colors'][12], 
                                           adata_gene.uns['cell_type_colors'][5]]

In [ ]:
sc.pl.spatial(adata_gene, color='cell_type_final', library_id=sample_id, img_key='hires_30rot', size=6, alpha_img=1.0, frameon=False)

In [ ]:
x_lower_roi1 = 8000
x_upper_roi1 = 13000
y_lower_roi1 = 15000
y_upper_roi1 = 28000

cells_roi1 = (adata_gene.obsm['spatial_30_rot'][:, 0]>=x_lower_roi1) * \
(adata_gene.obsm['spatial_30_rot'][:, 0]<=x_upper_roi1) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]>=y_lower_roi1) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]<=y_upper_roi1)

In [ ]:
x_lower_roi2 = 32000
x_upper_roi2 = 37000
y_lower_roi2 = 14000
y_upper_roi2 = 27000

cells_roi2 = (adata_gene.obsm['spatial_30_rot'][:, 0]>=x_lower_roi2) * \
(adata_gene.obsm['spatial_30_rot'][:, 0]<=x_upper_roi2) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]>=y_lower_roi2) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]<=y_upper_roi2)

In [ ]:
adata_gene.uns['cell_type_colors'] = ['#1f77b4',
 '#ff7f0e',
 '#279e68',
 '#d62728',
 '#aa40fc',
 '#66FF00',#'#8c564b',
 '#e377c2',
 '#b5bd61',
 '#17becf',
 '#aec7e8',
 '#ffbb78',
 '#98df8a',
 '#ff9896',
 '#c5b0d5',
 '#fcf4a3']

In [ ]:
adata_msi.uns['spatial'][sample_id]['images']['hires_30rot'] = img_rotated * 0.5
adata_gene.uns['spatial'][sample_id]['images']['hires_30rot'] = img_rotated * 0.5

In [ ]:
adata_gene.obsm['spatial'] = adata_gene.obsm['spatial_30_rot'].copy()
sc.pl.spatial(adata_gene, color='cell_type_final', library_id=sample_id, img_key='hires_30rot', size=6, alpha_img=1.0, frameon=False)

In [ ]:
adata_msi2.var_names

In [ ]:
differ_genes = ['Grp', 'Tmem108', 'Chrd', 'Cerk', 'Barx2', 'Abcd2', 'Mob4', 'Fam241a', 'Kcng1',
               'Srgap1', 'Kcnj2', 'Tiam2', 'Ighm', 'Gadd45a', 'Cuta', 'Cdc42ep1', 'Plaat3', 'Myrf', 'Bmp3', 'Rasl10a', 'Prss12', 
               'Resp18', 'Elfn1', 'Pcbd1', 'Rarb', 'Slc4a11', 'Ntn1', 'Nell1', 'Barx2', 'Fkbp1b', 'Arg2', 'Ano2', 'Zfp941', 
                'Onecut2', 'Wdr83os', 'Dlx5', 
               'Fgf3', 'Gas7', 'Slc35d3', 'Igsf1', 'Enkur', 'Ido1', 'Marcksl1', 'Nol7', 'Hmgn2']

import scanpy as sc
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 16, 'axes.titlesize': 20, 'axes.labelsize': 18})
# It is recommended to use a subset or specific markers for readability 
# as differ_genes contains 48 genes.
sc.pl.dotplot(
    adata_gene, 
    var_names=differ_genes, 
    groupby='cell_type_final', 
    standard_scale='var',  # Scales expression 0 to 1 per gene for better visualization
    #cmap='viridis',
    title='Expression of marker genes across cell types'
)

In [ ]:
import numpy as np
import pandas as pd

# list of genes defined in your notebook (Cell 128)
# differ_genes = ['Grp', 'Tmem108', 'Chrd', ...] 
adata_gene_threshold = adata_gene.copy()
percentile = 25

for gene in differ_genes:
    if gene in adata_gene_threshold.var_names:
        # Extract expression values for the specific gene
        # We use .toarray() in case the matrix is sparse
        gene_values = adata_gene_threshold[:, gene].X.toarray().flatten()
        
        # Calculate the 5th percentile threshold
        threshold = np.percentile(gene_values, percentile)
        
        # Find indices of cells below this threshold
        low_indices = adata_gene_threshold[:, gene].X < threshold
        
        # Set those values to zero in the main matrix
        # Note: If using sparse matrices, this approach is efficient
        adata_gene_threshold.X[low_indices.flatten(), adata_gene_threshold.var_names == gene] = 0

print(f"Lower {percentile}% of cells for each gene in differ_genes have been set to zero.")

In [ ]:
plt.rcParams.update({'font.size': 16, 'axes.titlesize': 20, 'axes.labelsize': 18})
# It is recommended to use a subset or specific markers for readability 
# as differ_genes contains 48 genes.
sc.pl.dotplot(
    adata_gene_threshold, 
    var_names=differ_genes, 
    groupby='cell_type_final', 
    standard_scale='var',  # Scales expression 0 to 1 per gene for better visualization
    #cmap='viridis',
    title='Expression of marker genes across cell types'
)

In [ ]:
np.unique(adata_gene.obs['cell_type_final'])

In [ ]:
adata_msi_roi_sub1 = adata_msi[cells_roi1 * ((adata_gene.obs['cell_type_final']=='Microglia') + (adata_gene.obs['cell_type_final']=='Oligodendrocytes')), :].copy()
adata_msi_roi_sub2 = adata_msi[cells_roi2 * ((adata_gene.obs['cell_type_final']=='Microglia') + (adata_gene.obs['cell_type_final']=='Oligodendrocytes')), :].copy()

In [ ]:
adata_msi2.X = adata_msi2.layers['counts'].copy()
sc.pp.normalize_total(adata_msi2)
sc.pp.log1p(adata_msi2)

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

y_lower_roi3 = 17000
y_upper_roi3 = 24000

cells_roi3 = (adata_gene.obsm['spatial_30_rot'][:, 0]>=x_lower_roi1) * \
(adata_gene.obsm['spatial_30_rot'][:, 0]<=x_upper_roi1) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]>=y_lower_roi3) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]<=y_upper_roi3)

y_lower_roi4 = 16000
y_upper_roi4 = 23000

cells_roi4 = (adata_gene.obsm['spatial_30_rot'][:, 0]>=(x_lower_roi2-450)) * \
(adata_gene.obsm['spatial_30_rot'][:, 0]<=(x_upper_roi2)) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]>=y_lower_roi4) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]<=y_upper_roi4)

adata_msi_roi_sub3 = adata_msi[cells_roi3 * ((adata_gene.obs['cell_type_final']=='Microglia') + (adata_gene.obs['cell_type_final']=='Oligodendrocytes')), :].copy()
adata_msi_roi_sub4 = adata_msi[cells_roi4 * ((adata_gene.obs['cell_type_final']=='Microglia') + (adata_gene.obs['cell_type_final']=='Oligodendrocytes')), :].copy()

# Create a figure with 1 row and 2 columns
fig, ax = plt.subplots(2, 2, figsize=(5, 6))
plt.subplots_adjust(wspace=0.1, hspace=0.1)
fig = sc.pl.spatial(adata_gene[adata_msi_roi_sub3.obs_names, :], color='cell_type_final', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, ax=ax[0,0])
fig = sc.pl.spatial(adata_msi2[adata_msi_roi_sub3.obs_names, :], color='393.12703', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, vmin=0.05, vmax=0.8, ax=ax[0,1], colorbar_loc=None)
#fig = sc.pl.spatial(adata_msi2[adata_msi_roi_sub3.obs_names, :], color='535.22436', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.045, vcenter=0.09, vmax=0.12, ax=ax[0,1], colorbar_loc=None)
fig = sc.pl.spatial(adata_gene[adata_msi_roi_sub4.obs_names, :], color='cell_type_final', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, ax=ax[1,0])
fig = sc.pl.spatial(adata_msi2[adata_msi_roi_sub4.obs_names, :], color='393.12703', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, vmin=0.05, vmax=0.8, ax=ax[1,1], colorbar_loc=None)
#fig = sc.pl.spatial(adata_msi2[adata_msi_roi_sub4.obs_names, :], color='535.22436', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.045, vcenter=0.09, vmax=0.12, ax=ax[1,1], colorbar_loc=None)

In [ ]:
# Create the figure
fig, ax = plt.subplots(2, 2, figsize=(7, 9)) # Increased figsize for better visibility
plt.subplots_adjust(wspace=0.05, hspace=0.01)

# Define the common plotting arguments to ensure consistency
spatial_args = {
    'img_key': 'hires_30rot',
    'size': 6,
    'alpha_img': 1.0,
    'cmap': 'turbo',
    'frameon': False,
    'legend_loc': None,
    'title': '',
    'show': False,
    'return_fig': True,
    'colorbar_loc': None # Ensure colorbars are off for perfect sizing
}

# Plot 1: ROI 3 Gene
sc.pl.spatial(adata_gene[adata_msi_roi_sub3.obs_names, :], color='cell_type_final', ax=ax[0, 0], **spatial_args)
# Plot 2: ROI 3 Peak
sc.pl.spatial(adata_msi2[adata_msi_roi_sub3.obs_names, :], layer='counts', color='393.12703', ax=ax[0, 1], vmin=0.05, vmax=0.8, **spatial_args)
#sc.pl.spatial(adata_msi2[adata_msi_roi_sub3.obs_names, :], layer='counts', color='535.22436', ax=ax[0, 1], vmin=0.045, vmax=0.12, **spatial_args)
# Plot 3: ROI 4 Gene
sc.pl.spatial(adata_gene[adata_msi_roi_sub4.obs_names, :], color='cell_type_final', ax=ax[1, 0], **spatial_args)
# Plot 4: ROI 4 Peak
sc.pl.spatial(adata_msi2[adata_msi_roi_sub4.obs_names, :], layer='counts', color='393.12703', ax=ax[1, 1], vmin=0.05, vmax=0.8, **spatial_args)
#sc.pl.spatial(adata_msi2[adata_msi_roi_sub4.obs_names, :], layer='counts', color='535.22436', ax=ax[1, 1], vmin=0.045, vmax=0.12, **spatial_args)

# Force all axes to have the same aspect ratio and limits
for a in ax.flat:
    a.set_aspect('equal')
    # Optional: If you want them to have the exact same pixel dimensions, 
    # you can manually set x/y limits based on your ROI definitions
    #a.set_xlim(x_min, x_max) 
    #a.set_ylim(y_min, y_max)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Extract the data for both regions
roi1_values = np.array(adata_msi2[adata_msi_roi_sub3.obs_names, '393.12703'].layers['counts']).flatten()
roi2_values = np.array(adata_msi2[adata_msi_roi_sub4.obs_names, '393.12703'].layers['counts']).flatten()

# 2. Create a DataFrame for Seaborn
df = pd.DataFrame({
    'Intensity': np.concatenate([roi1_values, roi2_values]),
    'Region': ['ROI 1'] * len(roi1_values) + ['ROI 2'] * len(roi2_values)
})

# 3. Create the boxplot
plt.figure(figsize=(6, 6))

# 5. Add P-value annotation (bracket and text)
# Define coordinates for the bracket
x1, x2 = 0, 1
# Calculate a position slightly above the data
y_max = 0.35#df['Intensity'].max()
y, h, col = y_max + 0.005, 0.002, 'k'

# Plot the bracket lines
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# Add the p-value text
plt.text((x1+x2)*.5, y+h, f"p < .0001", ha='center', va='bottom', color=col, fontsize=20)

# showfliers=False removes the outlier points
# palette fills the boxes with color
sns.boxplot(data=df, x='Region', y='Intensity', palette='Set1', showfliers=False, width=0.5, legend=False)

# Remove the top and right frame lines
sns.despine()

plt.ylabel('Intensity', fontsize=20)
plt.xlabel('m/z 393.12703', fontsize=20)
#plt.title('Comparison of Peak 535.22436 across ROIs')
#plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.scatter(adata_gene.obsm['spatial'][:, 0], adata_gene.obsm['spatial'][:, 1], s = 0.1)

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

y_lower_roi3 = 28000
y_upper_roi3 = 32000

cells_roi3 = (adata_gene.obsm['spatial_30_rot'][:, 0]<=20000) * \
(adata_gene.obsm['spatial_30_rot'][:, 0]>=15000) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]>=y_lower_roi3) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]<=y_upper_roi3)

y_lower_roi4 = 28000
y_upper_roi4 = 32000

cells_roi4 = (adata_gene.obsm['spatial_30_rot'][:, 0]>=30000) * \
(adata_gene.obsm['spatial_30_rot'][:, 0]<=35000) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]>=y_lower_roi4) * \
(adata_gene.obsm['spatial_30_rot'][:, 1]<=y_upper_roi4)

adata_msi_roi_sub3 = adata_msi[cells_roi3 * ((adata_gene.obs['cell_type_final']=='Microglia') + (adata_gene.obs['cell_type_final']=='Oligodendrocytes')), :].copy() # + (adata_gene.obs['cell_type']=='Mature/Stressed Oligodendrocytes')
adata_msi_roi_sub4 = adata_msi[cells_roi4 * ((adata_gene.obs['cell_type_final']=='Microglia') + (adata_gene.obs['cell_type_final']=='Oligodendrocytes')), :].copy() # + (adata_gene.obs['cell_type']=='Mature/Stressed Oligodendrocytes')

# Create a figure with 1 row and 2 columns
fig, ax = plt.subplots(2, 2, figsize=(5, 6))
plt.subplots_adjust(wspace=0.1, hspace=0.1)
fig = sc.pl.spatial(adata_gene[adata_msi_roi_sub3.obs_names, :], color='cell_type_final', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, ax=ax[0,0])
fig = sc.pl.spatial(adata_msi2[adata_msi_roi_sub3.obs_names, :], color='535.22436', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.05, vmax=0.8, ax=ax[0,1], colorbar_loc=None)
fig = sc.pl.spatial(adata_gene[adata_msi_roi_sub4.obs_names, :], color='cell_type_final', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, ax=ax[1,0])
fig = sc.pl.spatial(adata_msi2[adata_msi_roi_sub4.obs_names, :], color='535.22436', layer='counts', img_key='hires_30rot', size=6, alpha_img=1.0, cmap='turbo',frameon=False, legend_loc=None, title='', show=False, return_fig=True, vmin=0.05, vmax=0.8, ax=ax[1,1], colorbar_loc=None)

In [ ]:
# Create the figure
fig, ax = plt.subplots(2, 2, figsize=(8, 8)) # Increased figsize for better visibility
plt.subplots_adjust(wspace=0.05, hspace=0.05)

# Define the common plotting arguments to ensure consistency
spatial_args = {
    'img_key': 'hires_30rot',
    'size': 6,
    'alpha_img': 1.0,
    'cmap': 'turbo',
    'frameon': False,
    'legend_loc': None,
    'title': '',
    'show': False,
    'return_fig': True,
    'colorbar_loc': None # Ensure colorbars are off for perfect sizing
}

# Plot 1: ROI 3 Gene
sc.pl.spatial(adata_gene[adata_msi_roi_sub3.obs_names, :], color='cell_type_final', ax=ax[0, 0], **spatial_args)
# Plot 2: ROI 3 Peak
sc.pl.spatial(adata_msi2[adata_msi_roi_sub3.obs_names, :], layer='counts', color='393.12703', ax=ax[0, 1], vmin=0.05, vmax=0.8, **spatial_args)
#sc.pl.spatial(adata_msi2[adata_msi_roi_sub3.obs_names, :], layer='counts', color='535.22436', ax=ax[0, 1], vmin=0.045, vmax=0.12, **spatial_args)
# Plot 3: ROI 4 Gene
sc.pl.spatial(adata_gene[adata_msi_roi_sub4.obs_names, :], color='cell_type_final', ax=ax[1, 0], **spatial_args)
# Plot 4: ROI 4 Peak
sc.pl.spatial(adata_msi2[adata_msi_roi_sub4.obs_names, :], layer='counts', color='393.12703', ax=ax[1, 1], vmin=0.05, vmax=0.8, **spatial_args)
#sc.pl.spatial(adata_msi2[adata_msi_roi_sub4.obs_names, :], layer='counts', color='535.22436', ax=ax[1, 1], vmin=0.045, vmax=0.12, **spatial_args)

# Force all axes to have the same aspect ratio and limits
for a in ax.flat:
    a.set_aspect('equal')
    # Optional: If you want them to have the exact same pixel dimensions, 
    # you can manually set x/y limits based on your ROI definitions
    #a.set_xlim(x_min, x_max) 
    #a.set_ylim(y_min, y_max)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Extract the data for both regions
roi1_values = np.array(adata_msi2[adata_msi_roi_sub3.obs_names, '393.12703'].layers['counts']).flatten()
roi2_values = np.array(adata_msi2[adata_msi_roi_sub4.obs_names, '393.12703'].layers['counts']).flatten()

# 2. Create a DataFrame for Seaborn
df = pd.DataFrame({
    'Intensity': np.concatenate([roi1_values, roi2_values]),
    'Region': ['ROI 1'] * len(roi1_values) + ['ROI 2'] * len(roi2_values)
})

# 3. Create the boxplot
plt.figure(figsize=(6, 6))

# 5. Add P-value annotation (bracket and text)
# Define coordinates for the bracket
x1, x2 = 0, 1
# Calculate a position slightly above the data
y_max = 0.35#df['Intensity'].max()
y, h, col = y_max + 0.005, 0.002, 'k'

# Plot the bracket lines
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# Add the p-value text
plt.text((x1+x2)*.5, y+h, f"p < .0001", ha='center', va='bottom', color=col, fontsize=20)

# showfliers=False removes the outlier points
# palette fills the boxes with color
sns.boxplot(data=df, x='Region', y='Intensity', palette='Set1', showfliers=False, width=0.5, legend=False)

# Remove the top and right frame lines
sns.despine()

plt.ylabel('Intensity', fontsize=20)
plt.xlabel('m/z 393.12703', fontsize=20)
#plt.title('Comparison of Peak 535.22436 across ROIs')
#plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
from scipy import stats
for peak in ['393.12703']:
    print(peak)
    #t_stat, p_value = stats.ttest_ind(adata_msi2[adata_msi_roi_sub3.obs_names, peak].layers['counts'], adata_msi2[adata_msi_roi_sub4.obs_names, peak].layers['counts'])
    t_stat, p_value = stats.mannwhitneyu(adata_msi2[adata_msi_roi_sub3.obs_names, peak].layers['counts'], adata_msi2[adata_msi_roi_sub4.obs_names, peak].layers['counts'])

    print(f"T-statistic: {t_stat}")
    print(f"P-value: {p_value}")

In [ ]:
from scipy import stats
for peak in ['393.12703']:
    print(peak)
    t_stat, p_value = stats.mannwhitneyu(adata_msi2[adata_msi_roi_sub1.obs_names, peak].layers['counts'], adata_msi2[adata_msi_roi_sub2.obs_names, peak].layers['counts'])

    print(f"T-statistic: {t_stat}")
    print(f"P-value: {p_value}")

In [ ]:
adata_msi.uns['spatial'][sample_id]['images']['hires_30rot'] = img_rotated * 0.5
adata_gene.uns['spatial'][sample_id]['images']['hires_30rot'] = img_rotated * 0.5

In [ ]:
# Create the figure
fig, ax = plt.subplots(1, 10, figsize=(25, 8)) # Increased figsize for better visibility
plt.subplots_adjust(wspace=0.05, hspace=0.05)

# Define the common plotting arguments to ensure consistency
spatial_args = {
    'img_key': 'hires_30rot',
    'size': 6,
    'alpha_img': 1.0,
    'cmap': 'turbo',
    'frameon': False,
    'legend_loc': None,
    'title': '',
    'show': False,
    'return_fig': True,
    'colorbar_loc': None # Ensure colorbars are off for perfect sizing
}

sc.pl.spatial(adata_gene[cells_roi1, :], color='cell_type_final', library_id=sample_id, ax=ax[0], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi1, :], color='674.2805', layer='counts', vmin=0, vmax=3.5, ax=ax[1], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi1, :], color='444.20715', layer='counts', vmin=0.1, vmax=1.0, ax=ax[2], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi1, :], color='393.12703', layer='counts', vmin=0.1, vmax = 0.8, ax=ax[3], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi1, :], color='421.19136', layer='counts', vmin=0.02, vmax = 1.5, ax=ax[4], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi1, :], color='423.18201', layer='counts', vmin=0.2, vmax = 1.0, ax=ax[5], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi1, :], color='435.20692', layer='counts', vmin=0.02,vmax=4.0, ax=ax[6], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi1, :], color='698.49196', layer='counts', vmin=0.02, vmax=1.0, ax=ax[7], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi1, :], color='371.17565', layer='counts', vmin=0.02, vmax=1.0, ax=ax[8], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi1, :], color='353.1652', layer='counts', vmin=0.1, vmax=0.7, ax=ax[9], **spatial_args)

In [ ]:
# Create the figure
fig, ax = plt.subplots(1, 10, figsize=(25, 8)) # Increased figsize for better visibility
plt.subplots_adjust(wspace=0.05, hspace=0.05)

# Define the common plotting arguments to ensure consistency
spatial_args = {
    'img_key': 'hires_30rot',
    'size': 6,
    'alpha_img': 1.0,
    'cmap': 'turbo',
    'frameon': False,
    'legend_loc': None,
    'title': '',
    'show': False,
    'return_fig': True,
    'colorbar_loc': None # Ensure colorbars are off for perfect sizing
}

sc.pl.spatial(adata_gene[cells_roi2, :], color='cell_type', library_id=sample_id, ax=ax[0], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi2, :], color='674.2805', layer='counts', vmin=0, vmax=3.5, ax=ax[1], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi2, :], color='444.20715', layer='counts', vmin=0.1, vmax=1.0, ax=ax[2], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi2, :], color='393.12703', layer='counts', vmin=0.1, vmax = 0.8, ax=ax[3], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi2, :], color='421.19136', layer='counts', vmin=0.02, vmax = 1.5, ax=ax[4], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi2, :], color='423.18201', layer='counts', vmin=0.2, vmax = 1.0, ax=ax[5], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi2, :], color='435.20692', layer='counts', vmin=0.02,vmax=4.0, ax=ax[6], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi2, :], color='698.49196', layer='counts', vmin=0.02, vmax=1.0, ax=ax[7], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi2, :], color='371.17565', layer='counts', vmin=0.02, vmax=1.0, ax=ax[8], **spatial_args)
fig = sc.pl.spatial(adata_msi2[cells_roi2, :], color='353.1652', layer='counts', vmin=0.1, vmax=0.7, ax=ax[9], **spatial_args)

In [ ]:
adata_gene.uns['cell_type_colors']

In [ ]:
adata_gene.obs['cell_type_final'].cat.categories

In [ ]:
sample_id2

In [ ]:
adata_msi2.write_h5ad(f'{sample_id2}_MALDI_adata.h5ad')

In [ ]:
adata_gene.write_h5ad(f'{sample_id2}_RNA_joint_cluster_with_hres_image.h5ad')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# 1. Select the specific peaks
selected_peaks = ['674.2805', '421.19136', '435.20692']#, '371.17565']
peak_labels = ['DA (DD)', 'DA', '3-MT']#, 'GABA']

# 2. Setup a single-axes plot
plt.figure(figsize=(12, 8))

# 3. Get the standard Scanpy colors for cell types
cell_type_categories = adata_gene.obs['cell_type_final'].cat.categories
cell_type_colors = adata_gene.uns['cell_type_final_colors']
color_map = dict(zip(cell_type_categories, cell_type_colors))

# 4. Loop through each peak and plot onto the SAME axes
for i, peak_id in enumerate(selected_peaks):
    # Calculate Intensity and identify top 5% spots
    peak_intensity = adata_msi2[:, peak_id].layers['counts'].toarray().flatten()
    is_high = peak_intensity > np.percentile(peak_intensity, 95)
    
    # Generate high_perc for this specific peak
    high_counts = adata_gene.obs[is_high]['cell_type_final'].value_counts()
    high_perc = (high_counts / high_counts.sum()) * 100
    
    # Plot the stacked bar at position i
    bottom = 0
    x_label = peak_labels[i]
    
    for cell_type in high_perc.index:
        plt.bar(x_label, high_perc[cell_type], bottom=bottom, 
                color=color_map[cell_type], width=0.6)
        
        # Add text labels for segments > 5%
        '''
        if high_perc[cell_type] > 5:
            plt.text(x_label, bottom + (high_perc[cell_type]/2), 
                     f'{high_perc[cell_type]:.0f}%', ha='center', va='center', 
                     color='white', fontweight='bold', fontsize=11)
        '''
        bottom += high_perc[cell_type]

# 5. Formatting the single plot
#plt.title('Cell Type Composition Across High-Intensity Peaks', fontsize=18)
plt.ylabel('Percentage (%)', fontsize=20)
plt.ylim(0, 100)
plt.xticks(fontsize=18)

# Add a single shared legend to the right
plt.legend(['Striatal/Border Cells', 'iSPN', 'dSPN', 'GABAergic Interneurons', 'Interneurons', 'Oligodendrocytes', 'Endothelial Cells', 
            'Glutamatergic Neurons', 'Developing Striatal Neurons', 'Deep Layer Cortical Neurons', 'Immune Associated Cells', 
            'Large Projecting Neurons', 'Astrocytes', 'Ependymal Cells', 'Microglia'], loc='center left', bbox_to_anchor=(1.0, 0.5), frameon=False)

sns.despine()
plt.tight_layout()
plt.show()